<a href="https://colab.research.google.com/github/Pascingo/CTA-media-analysis/blob/Ga%C3%ABl/Stage_2.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Stage 2: Embedding Models Training and Analysis (Track 1: AI Governance)

In this notebook we train and compare embedding models on the AI Media Dataset. The selected track (AI Governance) is used as the focus case for the analysis. This first part prepares the data and trains a classical Word2Vec model:

1. Setup and download of the dataset
2. Loading and cleaning the data (time frame, text cleaning, duplicates, language filter)
3. Train / validation / test split (shared by the whole team)
4. Sentences and tokens
5. Detection of multi-word terms (phrases)
6. Training of Word2Vec
7. Nearest neighbours of AI Governance terms
8. Document vectors and separation of governance articles

All results that take long to compute (cleaned data, split, phrase model, Word2Vec model) are stored in the team's Google Drive, so the slow steps only have to run once.

## 1. Setup (Contributor: TBD)

We put all imports and settings at the top. The time frame and the folder structure are the same as in Stage 1.

**Cache:** With `USE_CACHE = True`, results that already exist in the Drive folder `CTA-media-analysis/stage2` are loaded instead of computed again. Set it to `False` to recompute everything (e.g. after changing the cleaning). The Drive folder must be available in "My Drive" (for a shared folder: add a shortcut to "My Drive").

In [1]:
!pip install -q gensim ftfy

import ast
import glob
import html
import os
import re
import shutil
import unicodedata
from collections import Counter

import gdown
import nltk
import numpy as np
import pandas as pd
import plotly.express as px
from ftfy import fix_text
from gensim.models import Word2Vec
from gensim.models.phrases import FrozenPhrases, Phrases
from google.colab import drive
from nltk.corpus import stopwords
from sklearn.linear_model import LogisticRegression
from sklearn.manifold import TSNE
from sklearn.metrics import silhouette_score
from sklearn.model_selection import cross_val_score, train_test_split
from tqdm.auto import tqdm

tqdm.pandas()

# Settings (same time frame as in Stage 1)
TEXT_COLUMN = "content"
START_DATE = "2024-09-01"
END_DATE = "2026-08-31"
RANDOM_STATE = 42

# Load existing results from the cache instead of computing them again
USE_CACHE = True

# Local folder (same as in Stage 1)
DATA_FOLDER = "data"
os.makedirs(DATA_FOLDER, exist_ok=True)

# Fixed dataset version from Stage 1 (Google Drive file ID of ai_media_dataset_20260911.csv)
DATASET_DRIVE_ID = "1CVjm5_r8EmMPjaN9qghdUYoFGlRy7Sz6"
DATASET_FILE = os.path.join(DATA_FOLDER, "ai_media_dataset_20260911.csv")

# Stage 2 results, stored locally and in the team's Google Drive
STAGE2_DRIVE_FOLDER = "/content/drive/MyDrive/CTA-media-analysis/stage2"
CLEANED_FILE = os.path.join(DATA_FOLDER, "stage2_cleaned.parquet")
SPLITS_FILE = os.path.join(DATA_FOLDER, "splits.csv")
BIGRAM_FILE = os.path.join(DATA_FOLDER, "bigram.phraser")
W2V_FILE = os.path.join(DATA_FOLDER, "w2v_train.model")

drive.mount("/content/drive")
os.makedirs(STAGE2_DRIVE_FOLDER, exist_ok=True)


def save_to_drive(path):
    """Copy a result to the Drive folder. gensim stores large models in several files (model + .npy), so we copy all of them."""
    for file in glob.glob(path + "*"):
        shutil.copy(file, STAGE2_DRIVE_FOLDER)


def restore_from_drive(path):
    """Copy a result from the Drive folder to DATA_FOLDER if it is not there yet. Returns True if the file exists."""
    if not os.path.exists(path):
        pattern = os.path.join(STAGE2_DRIVE_FOLDER, os.path.basename(path)) + "*"
        for file in glob.glob(pattern):
            shutil.copy(file, DATA_FOLDER)
    return os.path.exists(path)

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 27.8/27.8 MB 18.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 44.8/44.8 kB 1.9 MB/s eta 0:00:00
Mounted at /content/drive


### Download of the dataset

We use the same fixed dataset version as in Stage 1 (2026-09-11). Only this file is downloaded from the public Google Drive folder, and only if it is not already in the folder `data`.

In [2]:
if not os.path.exists(DATASET_FILE):
    gdown.download(id=DATASET_DRIVE_ID, output=DATASET_FILE, quiet=False)

if not os.path.exists(DATASET_FILE):
    raise FileNotFoundError(f"{DATASET_FILE} not found. Please check the Google Drive folder.")

print("Using file:", DATASET_FILE)

Downloading...
From (original): https://drive.google.com/uc?id=1CVjm5_r8EmMPjaN9qghdUYoFGlRy7Sz6
From (redirected): https://drive.google.com/uc?id=1CVjm5_r8EmMPjaN9qghdUYoFGlRy7Sz6&confirm=t&uuid=3d937c80-010a-4e78-ae73-3c0c37dfc1ed
To: /content/data/ai_media_dataset_20260911.csv
100%|██████████| 530M/530M [00:10<00:00, 49.2MB/s]

Using file: data/ai_media_dataset_20260911.csv


## 2. Loading and cleaning the data (Contributor: TBD)

In Stage 1 we cleaned the text for lemma-based methods (everything in lowercase, lemmatized). For Stage 2 we need two other versions of the text:

- **`base_text`** for the language models in the fine-tuning part: upper and lower case, punctuation and sentence structure stay, because LLMs are trained on normal text.
- **`w2v_text`** for Word2Vec: lowercase and only letters, digits and sentence punctuation, so that the same word always gets the same token.

The cleaning takes several minutes (mainly the repair of encoding errors) and is therefore cached. It has the following steps:

1. **Paragraphs:** The column `content` stores a Python list of paragraphs as a string (`"['Paragraph 1', 'Paragraph 2']"`). We convert it back to a list and join the paragraphs, otherwise the brackets and quotes would stay in the text.
2. **Encoding errors:** Some articles contain broken characters, e.g. `Weâ€™ve` instead of `We've` (we saw this in the NER results of Stage 1). `ftfy` repairs them before we remove the remaining special characters. A few product pages contain `â€�`, a quotation mark that cannot be repaired automatically; we replace it with `"`.
3. **Basic cleaning:** HTML tags and entities, URLs, typographic dashes and quotes. Some sources write `year' s` instead of `year's`; we join these again.
4. **Time frame:** Like in Stage 1 we only keep September 2024 to August 2026.
5. **Duplicates:** We remove duplicates after the cleaning, because two texts can only differ in HTML or encoding. We also remove duplicate URLs, so that the URL can be used as a unique ID of an article.
6. **Language:** The dataset also contains about 350 non-English articles, mainly German, Spanish, Portuguese and French (in a first run, the nearest neighbours of `open_source` were German words like `webanalyselosung_die`). For every article we compute the share of stop words of several languages among the first 300 words. An article is kept if at least 10% of these words are English stop words **and** the English share is higher than the share of every other language. A fixed threshold alone is not enough: English product pages with many lists and technical terms have only 15-20% English stop words, while Spanish or Portuguese pages with English navigation text reach up to 15%. Texts in non-Latin scripts (e.g. Japanese, Korean) are almost empty after the ASCII conversion and are removed by the 10% threshold. The language check is fast and therefore not part of the cache, so the threshold can be changed without running the cleaning again.
7. **Word2Vec text:** lowercase, possessive `'s` removed, hyphens between words replaced by a space ("london-based" -> "london based"), but kept between a name and a number ("gpt-4o" -> "gpt_4o"). The governance acronyms and names of laws are replaced with the same fixed tokens as in Stage 1 (e.g. "eu ai act" -> `eu_ai_act`). Many articles write the full name followed by the abbreviation, e.g. "General Data Protection Regulation (GDPR)". After the replacement this becomes "gdpr (gdpr)", so directly repeated words are reduced to one (otherwise `Phrases` would learn the token `gdpr_gdpr`).

In [3]:
MIN_ENGLISH_SHARE = 0.1
nltk.download("stopwords", quiet=True)
EN_STOP = set(stopwords.words("english"))

# Languages that occur in the dataset (and related ones), to compare their stop word shares with English
OTHER_LANGUAGES = ["german", "french", "spanish", "portuguese", "italian", "dutch", "indonesian",
                   "romanian", "swedish", "danish", "norwegian", "finnish", "turkish", "hungarian"]
OTHER_STOP = {lang: set(stopwords.words(lang)) for lang in OTHER_LANGUAGES}

# Same governance terms as in Stage 1 ("eu ai act" has to be replaced before "ai act")
ACRONYM_MAP = {
    r"\beu ai act\b": "eu_ai_act",
    r"\bai act\b": "ai_act",
    r"\bgeneral data protection regulation\b": "gdpr",
    r"\bfederal trade commission\b": "ftc",
    r"\blarge language models?\b": "llm",
    r"\bnational institute of standards and technology\b": "nist",
    r"\bgeneral purpose ai\b": "gpai",
    r"\bu\.s\.(?=\W|$)|\bunited states\b": "usa",
    r"\beuropean union\b": "eu",
    r"\bartificial intelligence\b": "ai",
}
ACRONYM_PATTERNS = [(re.compile(p), token) for p, token in ACRONYM_MAP.items()]


def parse_paragraphs(raw):
    """Turn the string "['Paragraph 1', 'Paragraph 2']" into one text."""
    if not isinstance(raw, str):
        return ""
    try:
        paragraphs = ast.literal_eval(raw)
        if isinstance(paragraphs, list):
            return " ".join(str(p) for p in paragraphs)
    except (ValueError, SyntaxError):
        pass
    return raw.strip("[]")


def base_clean(text):
    """Cleaning for all models: keeps upper and lower case and punctuation."""
    text = fix_text(text)                                        # "Weâ€™ve" -> "We've"
    text = text.replace("\u00e2\u20ac\ufffd", '"')                     # "â€�" (quote that ftfy cannot repair)
    text = html.unescape(text)                                   # "&amp;" -> "&"
    text = re.sub(r"<[^>]+>", " ", text)                         # HTML tags
    text = re.sub(r"https?://\S+|www\.\S+", " ", text)          # URLs
    text = re.sub(r"[\u2010-\u2015]", "-", text)                 # all dash variants -> "-"
    text = re.sub(r"[\u2018\u2019]", "'", text)                  # typographic apostrophes -> '
    text = re.sub(r"[\u201c\u201d]", '"', text)                  # typographic quotes -> "
    text = re.sub(r"(\w)\s*'\s+(s|t|re|ve|ll|d|m)\b", r"\1'\2", text)  # "year' s" -> "year's"
    text = unicodedata.normalize("NFKD", text).encode("ascii", "ignore").decode()
    return re.sub(r"\s+", " ", text).strip()


def w2v_clean(text):
    """Additional cleaning for Word2Vec."""
    text = text.lower()
    text = re.sub(r"'s\b", "", text)                                           # openai's -> openai
    text = re.sub(r"(?<=[a-z])-(?=[a-z])", " ", text)                          # london-based -> london based
    text = re.sub(r"(?<=[a-z0-9])-(?=[0-9])|(?<=[0-9])-(?=[a-z])", "_", text)  # gpt-4o -> gpt_4o
    text = text.replace("/", " ")
    for pattern, token in ACRONYM_PATTERNS:                                    # eu ai act -> eu_ai_act
        text = pattern.sub(token, text)
    text = re.sub(r"\b(\w+)(?:\W+\1\b)+", r"\1", text)                     # "gdpr ( gdpr)" -> "gdpr"
    text = re.sub(r"[^a-z0-9_\s.,!?]", " ", text)                               # other symbols
    return re.sub(r"\s+", " ", text).strip()


def language_shares(text, n_words=300):
    """Share of English stop words and the highest share of another language among the first n_words words."""
    words = re.findall(r"[a-z]+", text[:5000].lower())[:n_words]
    if not words:
        return 0.0, 0.0
    english = sum(word in EN_STOP for word in words) / len(words)
    other = max(sum(word in stop for word in words) for stop in OTHER_STOP.values()) / len(words)
    return english, other


def is_english_text(text):
    english, other = language_shares(text)
    return english >= MIN_ENGLISH_SHARE and english >= other


# Small checks of the cleaning functions
assert parse_paragraphs("['First part.', 'Second part.']") == "First part. Second part."
assert base_clean("We\u2019ve seen the <b>EU</b> year\u2019 s plan") == "We've seen the EU year's plan"
assert w2v_clean("The EU AI Act and OpenAI's GPT-4o, a London-based U.S. firm") == "the eu_ai_act and openai gpt_4o, a london based usa firm"
assert w2v_clean("The General Data Protection Regulation (GDPR) applies") == "the gdpr applies"
assert is_english_text("This is a text about the rules for AI in the EU")
assert not is_english_text("Die Webanalyseloesung ist datenschutzfreundlich und wird von vielen Firmen in der EU genutzt")
assert not is_english_text("La computacion en la nube es uno de los avances mas importantes de la economia europea")
print("Cleaning tests passed.")

Cleaning tests passed.


In [4]:
if USE_CACHE and restore_from_drive(CLEANED_FILE):
    df_all = pd.read_parquet(CLEANED_FILE)
    print(f"Loaded cleaned data from cache: {df_all.shape}")
else:
    df_all = pd.read_csv(DATASET_FILE, usecols=["title", "date", "content", "domain", "url", "tags"])
    print(f"Original dataset: {df_all.shape}")

    # Time frame
    df_all["date"] = pd.to_datetime(df_all["date"], errors="coerce")
    df_all = df_all[(df_all["date"] >= START_DATE) & (df_all["date"] <= END_DATE)].copy()
    print(f"After date filter ({START_DATE} to {END_DATE}): {df_all.shape}")

    # Paragraphs and basic cleaning
    df_all["base_text"] = df_all[TEXT_COLUMN].progress_apply(lambda raw: base_clean(parse_paragraphs(raw)))
    df_all = df_all[df_all["base_text"].str.len() > 0]

    # Duplicates (after cleaning) and duplicate URLs
    df_all = df_all.drop_duplicates(subset=["base_text"]).drop_duplicates(subset=["url"])
    print(f"After removing empty texts and duplicates: {df_all.shape}")

    # Word2Vec text (computed for all articles, the language filter follows below)
    df_all["w2v_text"] = df_all["base_text"].progress_apply(w2v_clean)

    df_all.to_parquet(CLEANED_FILE)
    save_to_drive(CLEANED_FILE)
    print(f"Saved '{CLEANED_FILE}' (also in Google Drive).")

# Language filter (not cached, takes about one minute)
shares = df_all["base_text"].progress_apply(language_shares)
df_all["english_share"] = [english for english, _ in shares]
df_all["other_share"] = [other for _, other in shares]
is_english = (df_all["english_share"] >= MIN_ENGLISH_SHARE) & (df_all["english_share"] >= df_all["other_share"])

removed = df_all.loc[~is_english, ["title", "domain", "english_share", "other_share"]]
print(f"\nNon-English articles removed: {len(removed)} ({len(removed) / len(df_all):.1%})")
print("Most frequent domains of the removed articles:")
display(removed["domain"].value_counts().head(10))
print("Removed articles with the highest English share (closest to the decision):")
display(removed.sort_values("english_share", ascending=False).head(10))

df = df_all[is_english].copy()
assert df["url"].is_unique, "URLs must be unique, they are used as article ID."
print(f"Final corpus: {df.shape}")
df[["title", "base_text", "w2v_text"]].head(3)

Original dataset: (47751, 6)
After date filter (2024-09-01 to 2026-08-31): (47508, 6)


  0%|          | 0/47508 [00:00<?, ?it/s]

After removing empty texts and duplicates: (47437, 7)


  0%|          | 0/47437 [00:00<?, ?it/s]

  0%|          | 0/47437 [00:00<?, ?it/s]

Saved 'data/stage2_cleaned.parquet' (also in Google Drive).

Non-English articles removed: 384 (0.8%)


,title,domain,english_share
4539,Fitness App Market to Expand by USD 55.86 Bill...,aijourn,0.196667
10279,7 Must-Have Features for Custom Software Devel...,netguru,0.193333
32775,Agentic & MCP Security,hiddenlayer,0.193333
18638,Lukas Brasiskis on Pierre Huyghe,e-flux,0.193333
46405,"Data, Context & RAG Lineage Governance for Ent...",dev,0.193333
45096,ViTel,tataelxsi,0.193333
31919,AI Agents for Order to Fulfillment,zbrain,0.193333
12211,Python 3.9.21 Crash with TensorFlow 2.10.1 Win...,ai.google,0.193333
40283,Top Cross-Platform Mobile App Development Comp...,tech-stack,0.190000
46620,HiddenLayer Introduces Agent Harness Security ...,hiddenlayer,0.190000


Final corpus: (47053, 9)


,title,base_text,w2v_text
0,Top 10 AI Companies in UK & Europe,With the field of AI accelerating all the time...,with the field of ai accelerating all the time...
1,“ Winner ” Takes Political Comedy Seriously,"This was in 2017, and she believed that the pu...","this was in 2017, and she believed that the pu..."
2,Optimizing Sustainable Construction with Smart...,A new study is shedding light on the latest in...,a new study is shedding light on the latest in...


The second table shows the removed articles with the highest share of English stop words, i.e. the cases closest to the decision. In our run these were mainly pages with an English beginning and a longer text in another language. If clearly English articles appear here, the rule should be adjusted.

## 3. Train / validation / test split (Contributor: TBD)

We split the articles into 80% training, 10% validation and 10% test data. The split is stratified by month, so every month has the same share in all three parts. This is important because the topics change over time (Stage 1): without stratification, for example, the test data could contain more articles about AI agents than the training data.

The split is done on the level of whole articles, so sentences of one article can never be in training and test data at the same time.

**The whole team uses the same split:** it is saved with the URL as a stable article ID in `splits.csv` in the Drive folder. If the file exists, it is loaded instead of creating a new split. The DataFrame index is not used as ID, because it changes as soon as the cleaning changes. If the cleaning is changed, `splits.csv` has to be deleted and created again, and all models have to be trained again on the new split.

In [5]:
df["month"] = df["date"].dt.to_period("M").astype(str)

if USE_CACHE and restore_from_drive(SPLITS_FILE):
    splits = pd.read_csv(SPLITS_FILE)
    df["split"] = df["url"].map(splits.set_index("url")["split"])
    missing = df["split"].isna().sum()
    if missing:
        raise ValueError(
            f"{missing} articles are not in {SPLITS_FILE}. The cleaning has changed: "
            "delete splits.csv (locally and in Drive) and create the split again."
        )
    print(f"Loaded split from '{SPLITS_FILE}'.")
else:
    train_ids, temp_ids = train_test_split(
        df.index, test_size=0.2, stratify=df["month"], random_state=RANDOM_STATE
    )
    val_ids, test_ids = train_test_split(
        temp_ids, test_size=0.5, stratify=df.loc[temp_ids, "month"], random_state=RANDOM_STATE
    )
    df["split"] = "train"
    df.loc[val_ids, "split"] = "val"
    df.loc[test_ids, "split"] = "test"

    df[["url", "date", "split"]].to_csv(SPLITS_FILE, index=False)
    save_to_drive(SPLITS_FILE)
    print(f"Created new split and saved it in '{SPLITS_FILE}' (also in Google Drive).")

print(df["split"].value_counts())

# Check: share of each month in the three parts (should be almost equal)
month_shares = pd.crosstab(df["month"], df["split"], normalize="columns").round(3)
display(month_shares.head(6))

Created new split and saved it in 'data/splits.csv' (also in Google Drive).
split
train    37642
test      4706
val       4705
Name: count, dtype: int64


split,test,train,val
month,,,
2024-09,0.020,0.021,0.021
2024-10,0.045,0.045,0.045
2024-11,0.037,0.037,0.037
2024-12,0.031,0.031,0.031
2025-01,0.052,0.052,0.052
2025-02,0.038,0.038,0.038


## 4. Sentences and tokens (Contributor: TBD)

Word2Vec learns from sentences, so we split every article into sentences (a sentence ends with `.`, `!` or `?` followed by a space, so "3.1" stays one token). In every sentence we keep:

- tokens with at least two characters and at least one letter. Pure numbers and number ranges like `0.001_0.003` carry no meaning for the embeddings, while names like `gpt_4o` stay.
- no stop words. We use the NLTK stop word list, because it only contains function words. The lists of gensim and scikit-learn also contain content words like "computer" or "system", which are important in texts about AI.

Sentences with less than three tokens are removed.

In [6]:
SENTENCE_SPLIT = re.compile(r"(?<=[.!?])\s+")
TOKEN_PATTERN = re.compile(r"[a-z0-9_.]+")
HAS_LETTER = re.compile(r"[a-z]")


def to_sentences(text):
    sentences = []
    for sentence in SENTENCE_SPLIT.split(text):
        tokens = [t.strip(".") for t in TOKEN_PATTERN.findall(sentence)]
        tokens = [t for t in tokens if len(t) > 1 and t not in EN_STOP and HAS_LETTER.search(t)]
        if len(tokens) >= 3:
            sentences.append(tokens)
    return sentences


# Check: numbers and number ranges are removed, sentences with less than 3 tokens are dropped
assert to_sentences("the eu_ai_act from 2024 applies to gpt_4o. values 0.001_0.003 are removed here.") == [
    ["eu_ai_act", "applies", "gpt_4o"]
]

df["sentences"] = df["w2v_text"].progress_apply(to_sentences)
train_sents = [s for doc in df.loc[df["split"] == "train", "sentences"] for s in doc]

n_tokens = sum(len(s) for s in train_sents)
print(f"{len(train_sents):,} sentences, {n_tokens:,} tokens in the training split")

  0%|          | 0/47053 [00:00<?, ?it/s]

3,043,165 sentences, 37,231,376 tokens in the training split


## 5. Multi-word terms (Contributor: TBD)

Many important terms consist of several words, e.g. "machine learning" or "national security". gensim `Phrases` finds word pairs that occur together much more often than by chance and joins them into one token (`machine_learning`). Two passes would also find terms with three words; one pass is enough for our purpose.

The phrase model is learned **only on the training split**, so no information from the validation and test data is used. Later, the same model is applied to the validation and test data.

In [7]:
if USE_CACHE and restore_from_drive(BIGRAM_FILE):
    bigram = FrozenPhrases.load(BIGRAM_FILE)
    print("Loaded phrase model from cache.")
else:
    bigram = Phrases(train_sents, min_count=10, threshold=10).freeze()
    bigram.save(BIGRAM_FILE)
    save_to_drive(BIGRAM_FILE)

train_phr = [bigram[s] for s in tqdm(train_sents)]

phrase_counts = Counter(t for s in train_phr for t in s if "_" in t)
print(f"{len(phrase_counts):,} different multi-word tokens")

# Check: the most frequent phrases related to AI Governance
governance_stems = ["regulat", "govern", "_act", "law", "bias", "privacy", "safety", "ethic", "complian"]
governance_phrases = [(p, n) for p, n in phrase_counts.most_common() if any(stem in p for stem in governance_stems)]
print(governance_phrases[:50])

  0%|          | 0/3043165 [00:00<?, ?it/s]

68,728 different multi-word tokens
['1.1_compliant', '19650_compliant', '2.0_compliant', '2.1_compliant', '2024_malawi', '25119_compliant', '26000_compliant', '26262_compliant', '2_compliant', '2pm_safety', '3.0_compliant', '3_compliant', '50_action', '800_155_compliant', '8_compliant', 'a16z_acting', 'academic_government', 'accordance_privacy', 'accountability_act', 'action_lawsuit', 'addressing_ethical', 'adhering_regulations', 'adopted_regulation', 'affirmative_action', 'african_governments', 'agencies_govern', 'agree_privacy', 'agreement_privacy', 'ai_act', 'algorithmic_bias', 'algorithmic_biases', 'aligning_activities', 'alkali_activated', 'alkali_activation', 'allied_governments', 'american_actor', 'aml_compliance', 'amplify_biases', 'analytics_cookielawinfo', 'anomalous_activity', 'antitrust_law', 'antitrust_laws', 'antitrust_lawsuit', 'antitrust_regulators', 'ap2_compliant', 'applicable_law', 'applicable_laws', 'apt_actors', 'asil_compliance', 'australian_actor', 'australian_go

## 6. Word2Vec (Contributor: TBD)

We train a skip-gram model (`sg=1`), because it learns better vectors for rare words than CBOW, and many governance terms are rare compared to general AI vocabulary.

| Parameter | Value | Reason |
|---|---|---|
| `vector_size` | 200 | common size for a corpus of about 40 million tokens |
| `window` | 5 | words up to 5 positions away count as context |
| `min_count` | 5 | words that occur less than 5 times get no vector |
| `negative` | 10 | negative sampling, recommended 5-20 for smaller corpora |
| `epochs` | 15 | more passes than the default (5), because the corpus is small for Word2Vec |

**Reproducibility:** With `workers=4` the training is about four times faster, but the results differ minimally between runs, because the threads process the sentences in a different order. Fully reproducible results would need `workers=1` and a fixed environment variable `PYTHONHASHSEED`. We accept the small differences and save the trained model in Drive, so all following analyses use the same model.

In [ ]:
if USE_CACHE and restore_from_drive(W2V_FILE):
    model = Word2Vec.load(W2V_FILE)
    print("Loaded Word2Vec model from cache.")
else:
    model = Word2Vec(
        sentences=train_phr,
        vector_size=200,
        window=5,
        min_count=5,
        sg=1,
        negative=10,
        epochs=15,
        workers=4,
        seed=RANDOM_STATE,
    )
    model.save(W2V_FILE)
    save_to_drive(W2V_FILE)

print(f"Vocabulary: {len(model.wv):,} words")

## 7. Nearest neighbours of AI Governance terms (Contributor: TBD)

A first check of the embeddings: if the nearest neighbours of a term are meaningful, the preprocessing and training work. If the neighbours are noise, the preprocessing is usually the reason. If a term is missing in the vocabulary, it is too rare or was written differently, e.g. joined with another word by `Phrases`.

In [ ]:
GOVERNANCE_TERMS = [
    "regulation", "eu_ai_act", "ai_act", "gdpr", "nist", "ftc",
    "accountability", "transparency", "liability", "oversight",
    "bias", "fairness", "discrimination", "privacy",
    "safety", "deepfake", "cybersecurity", "compliance",
]

rows = []
for term in GOVERNANCE_TERMS:
    if term not in model.wv:
        print(f"'{term}' is not in the vocabulary")
        continue
    neighbours = model.wv.most_similar(term, topn=5)
    rows.append({
        "term": term,
        "count": model.wv.get_vecattr(term, "count"),
        "nearest neighbours": ", ".join(f"{w} ({s:.2f})" for w, s in neighbours),
    })

display(pd.DataFrame(rows))

In [ ]:
SEED_TERMS = ["regulation", "eu_ai_act", "accountability", "bias", "privacy", "deepfake", "compliance"]

rows = []
for seed in SEED_TERMS:
    if seed not in model.wv:
        continue
    rows.append((seed, seed, "key term"))
    for word, _ in model.wv.most_similar(seed, topn=10):
        rows.append((word, seed, "neighbour"))

terms = pd.DataFrame(rows, columns=["word", "group", "kind"]).drop_duplicates("word")
vectors = np.array([model.wv[w] for w in terms["word"]])

coords = TSNE(n_components=2, perplexity=15, metric="cosine", init="pca",
              random_state=RANDOM_STATE).fit_transform(vectors)
terms["x"], terms["y"] = coords[:, 0], coords[:, 1]

fig = px.scatter(terms, x="x", y="y", color="group", symbol="kind", text="word",
                 hover_name="word", width=950, height=700,
                 title="Word2Vec: AI Governance terms and their nearest neighbours (t-SNE)")
fig.update_traces(textposition="top center", textfont_size=9)
fig.show()

## 8. Document vectors (Contributor: TBD)

To compare Word2Vec later with Doc2Vec and the fine-tuned language models, we need one vector per article. For Word2Vec we take the mean of all word vectors of an article.

We check how well these vectors separate AI Governance articles from the other articles. As long as the labels of the fine-tuning part are not available, we use the keyword filter from Stage 1 (at least 5 hits of the governance keywords) as label. This label is noisy, but it was created independently of the embeddings. A clustering of the same vectors (e.g. KMeans) would not be a meaningful label here, because the silhouette score would only measure how well KMeans fits its own clusters.

We only use articles from the validation and test split and compute two measures:

- **Silhouette score** (cosine): do the two groups form separate clusters? Averaged word vectors of all articles point in a very similar direction, because every article contains many general words. We therefore subtract the mean vector of the sample (centering) before computing the score. As reference we compute the score with randomly shuffled labels (should be about 0).
- **Linear separability:** a logistic regression with 5-fold cross-validation predicts the label from the vector (macro-F1). This shows whether the vectors contain information about the label, even if the groups do not form clusters. As reference we report the macro-F1 of always predicting the larger group.

In [ ]:
# Stage 1 governance filter (same keywords and threshold as in Stage 1)
GOVERNANCE_KEYWORDS = [
    "security", "accountability", "bias", "fairness",
    "regulatory", "regulation", "governance", "policy",
    "ethics", "compliance", "law",
]
MIN_KEYWORD_HITS = 5
N_SAMPLE = 3000

keyword_pattern = r"\b(?:" + "|".join(GOVERNANCE_KEYWORDS) + r")\b"
df["stage1_governance"] = df["base_text"].str.lower().str.count(keyword_pattern) >= MIN_KEYWORD_HITS


def doc_vector(sentences):
    """Mean of the word vectors of an article (with the same phrase model as in training)."""
    tokens = [t for s in sentences for t in bigram[s] if t in model.wv.key_to_index]
    if not tokens:
        return None
    return model.wv[tokens].mean(axis=0)


eval_docs = df[df["split"].isin(["val", "test"])]
sample = eval_docs.sample(n=min(N_SAMPLE, len(eval_docs)), random_state=RANDOM_STATE).copy()
sample["vec"] = sample["sentences"].apply(doc_vector)
sample = sample[sample["vec"].notna()].copy()
X = np.vstack(sample["vec"].values)

sample["label"] = np.where(sample["stage1_governance"], "governance (Stage 1 filter)", "other")
print(sample["label"].value_counts())

silhouette = silhouette_score(X, sample["label"], metric="cosine")
shuffled = sample["label"].sample(frac=1, random_state=RANDOM_STATE).to_numpy()
silhouette_random = silhouette_score(X, shuffled, metric="cosine")
print(f"\nSilhouette score: {silhouette:.3f} (random labels: {silhouette_random:.3f})")

coords = TSNE(n_components=2, perplexity=30, metric="cosine", init="pca",
              random_state=RANDOM_STATE).fit_transform(X)
sample["x"], sample["y"] = coords[:, 0], coords[:, 1]

fig = px.scatter(sample, x="x", y="y", color="label",
                 hover_data={"title": True, "date": True, "x": False, "y": False},
                 opacity=0.6, width=950, height=700,
                 title=f"Word2Vec document vectors (t-SNE, n={len(sample)}, silhouette={silhouette:.2f})")
fig.update_traces(marker_size=4)
fig.show()

The silhouette score ranges from -1 to +1: values close to 0 mean that the two groups strongly overlap, positive values mean that they form separate clusters. In a test run on the full data, the silhouette score was only slightly above the random reference (about 0.03 vs. 0.00), while the logistic regression reached a macro-F1 of about 0.77 (always the larger group: about 0.43). The vectors therefore contain clear information about the label, but governance articles do not form a separate cluster: they are spread across many topics (business, security, politics), as the topic model in Stage 1 already showed. In addition, the Stage 1 filter only counts keywords and also marks articles that are not about AI Governance. In the fine-tuning part, this analysis is repeated with the LLM-generated governance labels, which allows a fair comparison of Word2Vec, Doc2Vec and the fine-tuned models on the same data.

## 9. Fine-Tuning Compact Open Models (Contributor: TBD)

In this chapter we fine-tune compact open language models on a task that is directly relevant to our track, and compare them with the classical representations from the previous sections.

**Task: classification of AI Governance subtopics.** Every article gets exactly one label:

| Label | Content |
|---|---|
| `REGULATION_POLICY` | laws, regulation, government policy, standards bodies, court cases about AI (e.g. EU AI Act, executive orders, copyright lawsuits) |
| `SECURITY_SAFETY` | AI security and safety: attacks on or with AI systems, misuse, deepfakes, model safety and risks |
| `BIAS_FAIRNESS_ETHICS` | bias, discrimination, fairness, ethics, human rights |
| `ACCOUNTABILITY_TRANSPARENCY` | transparency, explainability, liability, audits, responsible-AI programmes and internal governance frameworks of organisations |
| `NOT_GOVERNANCE` | AI Governance is not the main topic of the article |

We chose this task for three reasons. It is an extrinsic task as required for Stage 2, it measures how well a model understands the selected track, and it repairs a weakness of Stage 1: the keyword filter marked many articles as "AI Governance" that are not about governance (e.g. "What Is Overfitting?" as bias/fairness). A model trained with the class `NOT_GOVERNANCE` can be used for a cleaner trend analysis.

**Procedure:**

1. Setup (GPU, libraries, settings)
2. Selection of 2,600 articles from the existing train / validation / test split
3. Labelling with a large language model (silver labels)
4. Manual check of the test labels
5. Baselines: TF-IDF and Word2Vec with logistic regression
6. Zero-shot baseline: the same open models without fine-tuning
7. Fine-tuning with QLoRA: Qwen3-1.7B and Qwen3-4B
8. Comparison of all models (quality, size, speed, GPU memory)
9. Embeddings of the fine-tuned model compared with Word2Vec
10. Trend analysis of the governance subtopics over the whole corpus

**Runtime:** This chapter needs a GPU runtime (Colab Pro: *Runtime > Change runtime type > L4 GPU* or *A100 GPU*). On an L4 GPU the whole chapter takes about 2-3 hours, most of it for the fine-tuning of Qwen3-4B and the classification of the whole corpus. All results (labels, LoRA adapters, predictions) are stored in Drive, so a second run only loads them.

### 9.1 Setup

The libraries `peft` (LoRA), `bitsandbytes` (4-bit quantisation) and `anthropic` (labelling) are not pre-installed in Colab. We also free the memory used by the sentences of the Word2Vec part; the few articles needed in this chapter are tokenised again.

**Model choice.** From the candidate models of the task description we use two sizes of the same family, **Qwen3-1.7B** and **Qwen3-4B**. Because both models share the architecture, tokenizer and training data, the comparison shows the effect of the model size alone. Other candidates (e.g. Apertus or Gemma) can be added to `MODELS`.

In [ ]:
!pip install -q peft bitsandbytes anthropic

import gc
import json
import threading
import time
from concurrent.futures import ThreadPoolExecutor

import anthropic
import matplotlib.pyplot as plt
import torch
import torch.nn.functional as F
import transformers
from datasets import Dataset
from google.colab import userdata
from peft import LoraConfig, PeftModel, TaskType, get_peft_model, prepare_model_for_kbit_training
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics import (ConfusionMatrixDisplay, accuracy_score, classification_report,
                             cohen_kappa_score, f1_score)
from transformers import (AutoModelForCausalLM, AutoModelForSequenceClassification, AutoTokenizer,
                          BitsAndBytesConfig, DataCollatorWithPadding, Trainer, TrainingArguments)

assert torch.cuda.is_available(), "Please select a GPU runtime (Runtime > Change runtime type > L4 or A100)."
print(f"GPU: {torch.cuda.get_device_name(0)} | transformers {transformers.__version__}")

# Free the memory of the Word2Vec part
for name in ["train_sents", "train_phr", "phrase_counts"]:
    globals().pop(name, None)
df = df.drop(columns=["sentences"], errors="ignore")
gc.collect()

# Results of this chapter are stored in Drive
FT_DRIVE_FOLDER = os.path.join(STAGE2_DRIVE_FOLDER, "finetuning")
os.makedirs(FT_DRIVE_FOLDER, exist_ok=True)

LABELS = ["REGULATION_POLICY", "SECURITY_SAFETY", "BIAS_FAIRNESS_ETHICS",
          "ACCOUNTABILITY_TRANSPARENCY", "NOT_GOVERNANCE"]
LABEL2ID = {label: i for i, label in enumerate(LABELS)}
ID2LABEL = dict(enumerate(LABELS))

# Open models (Hugging Face IDs)
MODELS = {
    "Qwen3-1.7B": "Qwen/Qwen3-1.7B",
    "Qwen3-4B": "Qwen/Qwen3-4B",
}

N_LABELLED = {"train": 2000, "val": 300, "test": 300}   # articles to label per split
GOVERNANCE_SHARE = 0.6    # share of articles selected from the Stage 1 governance filter
INPUT_WORDS = 400         # first words of an article given to the labelling LLM and to all models
MAX_LENGTH = 512          # maximum tokens per article for the open models
USE_4BIT = True           # QLoRA: base model in 4 bit, LoRA adapters in 16 bit
COMPUTE_DTYPE = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16

# transformers renamed "torch_dtype" to "dtype" in version 4.56; this works with old and new versions
TRANSFORMERS_VERSION = tuple(int(x) for x in transformers.__version__.split(".")[:2])
DTYPE_ARG = {"dtype": COMPUTE_DTYPE} if TRANSFORMERS_VERSION >= (4, 56) else {"torch_dtype": COMPUTE_DTYPE}

### 9.2 Selection of the articles to label

We label 2,000 training, 300 validation and 300 test articles. They are taken from the **same split** as the Word2Vec part, so test articles were never used for training of any model.

A random sample would contain only few governance articles (in Stage 1, only 26% of the articles passed the keyword filter, and many of them are not about governance). We therefore take 60% of the articles from the Stage 1 governance filter and 40% from the other articles. This gives enough examples of every governance class, while the class `NOT_GOVERNANCE` still contains many difficult cases (articles with governance keywords that are about something else). Because of this selection, the class distribution of the labelled data is not the distribution of the whole corpus.

All models get the same input: the title and the first 400 words of the cleaned text (`base_text`).

In [ ]:
assert "stage1_governance" in df.columns, "Please run section 8 first."


def model_input(row):
    """Title and the first INPUT_WORDS words of the article."""
    words = str(row["base_text"]).split()[:INPUT_WORDS]
    return f"Title: {row['title']}\n\n" + " ".join(words)


def sample_split(split, n):
    part = df[df["split"] == split]
    candidates = part[part["stage1_governance"]]
    gov = candidates.sample(n=min(int(n * GOVERNANCE_SHARE), len(candidates)), random_state=RANDOM_STATE)
    other = part[~part["stage1_governance"]].sample(n=n - len(gov), random_state=RANDOM_STATE)
    return pd.concat([gov, other])


labelled = pd.concat([sample_split(split, n) for split, n in N_LABELLED.items()])
labelled["input_text"] = labelled.apply(model_input, axis=1)

print(f"Articles to label: {len(labelled)}")
display(pd.crosstab(labelled["split"], labelled["stage1_governance"], margins=True))

### 9.3 Labelling with a large language model

Labelling 2,600 articles by hand is not feasible in this project. We therefore let a large language model assign the labels and check the test labels by hand in the next step (labels created like this are called *silver labels*, in contrast to manually created *gold labels*). The labelling model is **different from the open models that we fine-tune**, so the open models do not simply learn to copy their own answers.

**Model and settings.** We use **Claude Opus 5.5** (`claude-opus-5-5`) via the Anthropic API with low reasoning effort, because the task is a simple classification. The answer is returned as JSON with a fixed schema (*structured outputs*), so every answer contains exactly one of the five labels, a confidence and a short reason. If the model declines a request for safety reasons, the API automatically repeats it with a fallback model (`fallbacks: "default"`).

**Costs.** One request contains about 1,000 input tokens (instructions and article) and a few hundred output tokens, which gives roughly 20-25 USD for 2,600 articles with Claude Opus 5.5. A much cheaper alternative is Claude Haiku 5.5 (`LABEL_MODEL = "claude-haiku-5-5"`, below 1 USD), with possibly less accurate labels.

**API key:** In Colab, open *Secrets* (key symbol on the left), add `ANTHROPIC_API_KEY` and allow access for this notebook.

Each label is written to `llm_labels.jsonl` in Drive immediately. If the cell is interrupted or some requests fail, running it again only labels the missing articles.

In [ ]:
LABEL_MODEL = "claude-opus-5-5"     # cheaper alternative: "claude-haiku-5-5"
LABELS_FILE = os.path.join(FT_DRIVE_FOLDER, "llm_labels.jsonl")
N_WORKERS = 8                       # parallel requests

LABEL_SYSTEM_PROMPT = """You classify news articles about artificial intelligence for a research project on AI governance.

Assign exactly one label that describes the main topic of the article:

- REGULATION_POLICY: laws, regulation, government policy, standards bodies, or court cases concerning AI (e.g. EU AI Act, executive orders, AI copyright lawsuits, national AI strategies).
- SECURITY_SAFETY: security or safety of AI systems: attacks on or with AI, misuse, deepfakes, cybersecurity of AI, model safety and risks.
- BIAS_FAIRNESS_ETHICS: bias, discrimination, fairness, ethics, or human rights in relation to AI.
- ACCOUNTABILITY_TRANSPARENCY: transparency, explainability, liability, audits, responsible-AI programmes, or internal AI governance frameworks of organisations.
- NOT_GOVERNANCE: AI governance is not the main topic, e.g. product news, technical tutorials, business or market reports that only mention regulation, security, or ethics in passing.

Choose a governance label only if the topic is central to the article, not just mentioned. If several governance topics appear, choose the most prominent one. The article is cut after about 400 words; classify based on the visible text."""

LABEL_SCHEMA = {
    "type": "object",
    "properties": {
        "label": {"type": "string", "enum": LABELS},
        "confidence": {"type": "string", "enum": ["high", "medium", "low"]},
        "reason": {"type": "string", "description": "One short sentence."},
    },
    "required": ["label", "confidence", "reason"],
    "additionalProperties": False,
}

label_client = anthropic.Anthropic(api_key=userdata.get("ANTHROPIC_API_KEY"), max_retries=5)
write_lock = threading.Lock()


def label_article(text):
    """Returns {"label", "confidence", "reason"} or {"label": None, "error": ...}."""
    request = dict(
        model=LABEL_MODEL,
        max_tokens=4000,
        system=LABEL_SYSTEM_PROMPT,
        messages=[{"role": "user", "content": f"<article>\n{text}\n</article>"}],
        output_config={"effort": "low", "format": {"type": "json_schema", "schema": LABEL_SCHEMA}},
    )
    if LABEL_MODEL.startswith("claude-haiku"):
        response = label_client.messages.create(**request)   # Haiku has no server-side fallback
    else:
        response = label_client.beta.messages.create(
            **request, betas=["server-side-fallback-2026-07-01"], fallbacks="default"
        )
    if response.stop_reason != "end_turn":                    # e.g. "refusal" or "max_tokens"
        return {"label": None, "error": response.stop_reason}
    answer = next(block.text for block in response.content if block.type == "text")
    return json.loads(answer)


def load_label_file():
    labels = {}
    if os.path.exists(LABELS_FILE):
        with open(LABELS_FILE, encoding="utf-8") as f:
            for line in f:
                record = json.loads(line)
                if record.get("label") in LABELS:
                    labels[record["url"]] = record
    return labels


def label_and_save(row):
    try:
        result = label_article(row.input_text)
    except anthropic.BadRequestError as e:          # not retryable, e.g. invalid request
        result = {"label": None, "error": f"bad request: {e.message}"}
    except anthropic.APIStatusError as e:           # still failing after the SDK's retries
        result = {"label": None, "error": f"status {e.status_code}"}
    except anthropic.APIConnectionError:
        result = {"label": None, "error": "connection error"}
    record = {"url": row.url, **result}
    with write_lock, open(LABELS_FILE, "a", encoding="utf-8") as f:
        f.write(json.dumps(record) + "\n")
    return record


done = load_label_file()
todo = labelled[~labelled["url"].isin(done)]
print(f"Already labelled: {len(done)}, to label now: {len(todo)}")

if len(todo):
    with ThreadPoolExecutor(max_workers=N_WORKERS) as pool:
        failed = [r for r in tqdm(pool.map(label_and_save, todo.itertuples()), total=len(todo))
                  if r["label"] is None]
    print(f"Failed requests: {len(failed)} (run the cell again to retry them)")
    if failed:
        print(pd.Series([r["error"] for r in failed]).value_counts())

done = load_label_file()
labelled["llm_label"] = labelled["url"].map(lambda u: done.get(u, {}).get("label"))
labelled["llm_confidence"] = labelled["url"].map(lambda u: done.get(u, {}).get("confidence"))
labelled["llm_reason"] = labelled["url"].map(lambda u: done.get(u, {}).get("reason"))

print(f"\nArticles with label: {labelled['llm_label'].notna().sum()} of {len(labelled)}")
display(pd.crosstab(labelled["llm_label"], labelled["split"], margins=True))
display(pd.crosstab(labelled["llm_label"], labelled["stage1_governance"], normalize="columns").round(2))

The second table compares the LLM labels with the keyword filter from Stage 1: it shows which share of the articles selected by the filter are labelled as `NOT_GOVERNANCE`, i.e. how many false positives the filter produces.

### 9.4 Manual check of the test labels

The test set is used to compare all models, so its labels should be as reliable as possible. The cell exports the 300 test articles with the LLM labels to `test_labels_for_review.csv` in Drive.

**Procedure:** open the file (e.g. in Google Sheets), read title and text, and enter the correct label in the column `human_label` (one of the five labels). Save the result as `test_labels_reviewed.csv` in the same Drive folder and run the cell again. Then:

- the agreement between LLM and human labels is reported (accuracy and Cohen's kappa; kappa corrects for agreement by chance, values above 0.6 are usually regarded as substantial agreement),
- the human labels replace the LLM labels in the test set (gold labels).

As long as no reviewed file exists, the LLM labels are used and the results must be read as "agreement with the labelling model".

In [ ]:
REVIEW_FILE = os.path.join(FT_DRIVE_FOLDER, "test_labels_for_review.csv")
REVIEWED_FILE = os.path.join(FT_DRIVE_FOLDER, "test_labels_reviewed.csv")

if not os.path.exists(REVIEW_FILE):
    export = labelled.loc[labelled["split"] == "test",
                          ["url", "title", "input_text", "llm_label", "llm_confidence", "llm_reason"]].copy()
    export["human_label"] = ""
    export.to_csv(REVIEW_FILE, index=False)
    print(f"Exported {len(export)} test articles to '{REVIEW_FILE}'.")

labelled["gold_label"] = labelled["llm_label"]
labelled["label_source"] = "llm"

if os.path.exists(REVIEWED_FILE):
    reviewed = pd.read_csv(REVIEWED_FILE)
    reviewed["human_label"] = reviewed["human_label"].astype(str).str.strip().str.upper()
    reviewed = reviewed[reviewed["human_label"].isin(LABELS)].set_index("url")
    compare = labelled[labelled["url"].isin(reviewed.index) & labelled["llm_label"].notna()]
    human = compare["url"].map(reviewed["human_label"])

    print(f"Manually checked test articles: {len(compare)}")
    print(f"Agreement LLM - human: {(compare['llm_label'] == human).mean():.1%}, "
          f"Cohen's kappa: {cohen_kappa_score(compare['llm_label'], human):.3f}")
    display(pd.crosstab(compare["llm_label"], human, rownames=["LLM"], colnames=["human"]))

    mask = labelled["url"].isin(reviewed.index)
    labelled.loc[mask, "gold_label"] = labelled.loc[mask, "url"].map(reviewed["human_label"])
    labelled.loc[mask, "label_source"] = "human"
else:
    print("No reviewed file yet: the test set uses the LLM labels.")

### 9.5 Data for the models and classical baselines

We first train two simple models on the same labels. They show how much the language models add:

- **TF-IDF + logistic regression:** a bag-of-words model, the usual strong baseline for text classification.
- **Word2Vec + logistic regression:** the averaged Word2Vec vectors from section 8 as features.

Because the classes are not equally frequent, all models are compared with the **macro-F1 score** (the mean of the F1 scores of the five classes). With accuracy alone, a model that mostly predicts the frequent class `NOT_GOVERNANCE` would look good. For the same reason, the logistic regressions and the fine-tuning use class weights (rare classes count more in the loss).

All results are collected in the list `results` and compared in section 9.8.

In [ ]:
data = labelled[labelled["gold_label"].isin(LABELS)].copy()
data["label_id"] = data["gold_label"].map(LABEL2ID)
train_l, val_l, test_l = (data[data["split"] == s] for s in ["train", "val", "test"])
print(f"Train: {len(train_l)}, validation: {len(val_l)}, test: {len(test_l)}")
display(pd.crosstab(data["gold_label"], data["split"]))

# Class weights: inverse class frequency in the training data, limited to 10 so that a very rare class
# does not dominate the loss
class_counts = train_l["label_id"].value_counts().reindex(range(len(LABELS)), fill_value=0)
weights = (len(train_l) / (len(LABELS) * class_counts.clip(lower=1))).clip(upper=10)
class_weights = torch.tensor(weights.to_numpy(), dtype=torch.float32)
print("Class weights:", dict(zip(LABELS, class_weights.numpy().round(2))))

results = []
predictions = {}


def report(name, y_true, y_pred, **info):
    """Print and store the scores of a model on the test set."""
    macro_f1 = f1_score(y_true, y_pred, average="macro", labels=range(len(LABELS)), zero_division=0)
    accuracy = accuracy_score(y_true, y_pred)
    results.append({"model": name, "macro_f1": round(macro_f1, 3), "accuracy": round(accuracy, 3), **info})
    predictions[name] = np.asarray(y_pred)
    print(f"{name}: macro-F1 {macro_f1:.3f}, accuracy {accuracy:.3f}")


# --- TF-IDF + logistic regression ---
start = time.time()
tfidf = TfidfVectorizer(sublinear_tf=True, min_df=2, ngram_range=(1, 2), max_features=50000, stop_words="english")
X_train_tfidf = tfidf.fit_transform(train_l["input_text"])
tfidf_clf = LogisticRegression(max_iter=3000, class_weight="balanced").fit(X_train_tfidf, train_l["label_id"])
train_time = time.time() - start
start = time.time()
pred = tfidf_clf.predict(tfidf.transform(test_l["input_text"]))
report("TF-IDF + LogReg", test_l["label_id"], pred, train_min=round(train_time / 60, 2),
       infer_s_per_100=round((time.time() - start) / len(test_l) * 100, 3), params="-", trainable_params="-",
       peak_gpu_gb=0)

# --- Word2Vec + logistic regression (vectors from section 6/8) ---
def w2v_features(frame):
    vectors = [doc_vector(to_sentences(w2v_clean(text))) for text in frame["input_text"]]
    return np.vstack([v if v is not None else np.zeros(model.vector_size) for v in vectors])


start = time.time()
X_train_w2v = w2v_features(train_l)
w2v_clf = LogisticRegression(max_iter=3000, class_weight="balanced").fit(X_train_w2v, train_l["label_id"])
train_time = time.time() - start
start = time.time()
pred = w2v_clf.predict(w2v_features(test_l))
report("Word2Vec + LogReg", test_l["label_id"], pred, train_min=round(train_time / 60, 2),
       infer_s_per_100=round((time.time() - start) / len(test_l) * 100, 3), params="-", trainable_params="-",
       peak_gpu_gb=0)

### 9.6 Zero-shot baseline

Before fine-tuning, we test how well the open models solve the task **without training**: the model receives the same instructions as the labelling model and the article, and answers with one label. This shows how much of the performance comes from the fine-tuning and how much the models already know. We disable the "thinking" mode of Qwen3 (`enable_thinking=False`), so the model answers directly. If an answer contains no valid label, we count it as `NOT_GOVERNANCE` and report how often this happens.

In [ ]:
def clear_gpu():
    """Release GPU memory of models that are no longer referenced."""
    gc.collect()
    torch.cuda.empty_cache()


def parse_label(answer):
    answer = answer.upper()
    for label in LABELS:
        if label in answer:
            return LABEL2ID[label], True
    return LABEL2ID["NOT_GOVERNANCE"], False


def zero_shot(short_name, hf_name, frame, batch_size=16):
    tokenizer = AutoTokenizer.from_pretrained(hf_name, padding_side="left")
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    llm = AutoModelForCausalLM.from_pretrained(hf_name, **DTYPE_ARG, device_map={"": 0})
    llm.eval()

    prompts = [
        tokenizer.apply_chat_template(
            [{"role": "system", "content": LABEL_SYSTEM_PROMPT + "\n\nAnswer only with the label."},
             {"role": "user", "content": f"<article>\n{text}\n</article>"}],
            tokenize=False, add_generation_prompt=True, enable_thinking=False,
        )
        for text in frame["input_text"]
    ]

    torch.cuda.reset_peak_memory_stats()
    start = time.time()
    preds, valid = [], []
    for i in tqdm(range(0, len(prompts), batch_size), desc=f"Zero-shot {short_name}"):
        batch = tokenizer(prompts[i:i + batch_size], return_tensors="pt", padding=True).to(llm.device)
        with torch.no_grad():
            output = llm.generate(**batch, max_new_tokens=12, do_sample=False, pad_token_id=tokenizer.pad_token_id)
        answers = tokenizer.batch_decode(output[:, batch["input_ids"].shape[1]:], skip_special_tokens=True)
        for answer in answers:
            label_id, ok = parse_label(answer)
            preds.append(label_id)
            valid.append(ok)
    infer_time = time.time() - start

    print(f"Answers without a valid label: {len(valid) - sum(valid)} of {len(valid)}")
    report(f"{short_name} zero-shot", frame["label_id"], preds,
           train_min=0, infer_s_per_100=round(infer_time / len(frame) * 100, 2),
           params=f"{llm.num_parameters() / 1e9:.2f}B", trainable_params=0,
           peak_gpu_gb=round(torch.cuda.max_memory_allocated() / 1e9, 2))


for short_name, hf_name in MODELS.items():
    zero_shot(short_name, hf_name, test_l)
    clear_gpu()

### 9.7 Fine-tuning with QLoRA

A full fine-tuning would update all 1.7 or 4 billion parameters of a model, which needs far more GPU memory than Colab offers. We therefore use **QLoRA**, a parameter-efficient method:

- The weights of the base model are loaded in **4-bit** precision (NF4) and stay **frozen**. This reduces the memory of Qwen3-4B from about 8 GB to about 2.5 GB.
- Small trainable matrices (**LoRA adapters**, rank r = 16) are added to all attention and feed-forward layers. Only these adapters and the new classification layer are trained, which is less than 1-2% of all parameters.
- For classification we use the model with a **classification head** (`AutoModelForSequenceClassification`): the last hidden state of the last token is mapped to the five classes. Compared with generating the label as text, this gives probabilities for all classes and needs only one forward pass per article.

| Setting | Value | Reason |
|---|---|---|
| LoRA rank / alpha / dropout | 16 / 32 / 0.05 | common values for classification tasks |
| Learning rate | 2e-4 | usual for LoRA (much higher than for full fine-tuning) |
| Epochs | 3 | the validation score is checked after every epoch, the best epoch is kept |
| Batch size | 8 × 2 (gradient accumulation) = 16 | fits into the memory of an L4 GPU |
| Max. length | 512 tokens | title and first 400 words |

For every model we measure the training time, the peak GPU memory and the number of trainable parameters for the comparison of trade-offs. The trained adapters (about 30-70 MB) are saved in Drive; with `USE_CACHE = True` they are loaded instead of training again.

In [ ]:
LORA_TARGETS = ["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"]
EPOCHS = 3
LEARNING_RATE = 2e-4
BATCH_SIZE = 8
GRAD_ACCUM = 2


def load_classifier(hf_name):
    tokenizer = AutoTokenizer.from_pretrained(hf_name)
    if tokenizer.pad_token is None:
        tokenizer.pad_token = tokenizer.eos_token
    quantization = BitsAndBytesConfig(
        load_in_4bit=True, bnb_4bit_quant_type="nf4", bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=COMPUTE_DTYPE,
        llm_int8_skip_modules=["score"],          # keep the new classification layer in full precision
    ) if USE_4BIT else None
    classifier = AutoModelForSequenceClassification.from_pretrained(
        hf_name, num_labels=len(LABELS), id2label=ID2LABEL, label2id=LABEL2ID,
        quantization_config=quantization, **DTYPE_ARG, device_map={"": 0},
    )
    classifier.config.pad_token_id = tokenizer.pad_token_id
    return tokenizer, classifier


def to_dataset(frame, tokenizer):
    dataset = Dataset.from_dict({"text": frame["input_text"].tolist(), "labels": frame["label_id"].tolist()})
    return dataset.map(lambda batch: tokenizer(batch["text"], truncation=True, max_length=MAX_LENGTH),
                       batched=True, remove_columns=["text"])


class WeightedTrainer(Trainer):
    """Trainer with class weights in the cross-entropy loss."""

    def compute_loss(self, model, inputs, return_outputs=False, **kwargs):
        labels = inputs.pop("labels")
        outputs = model(**inputs)
        loss = F.cross_entropy(outputs.logits.float(), labels, weight=class_weights.to(outputs.logits.device))
        return (loss, outputs) if return_outputs else loss


def compute_metrics(eval_pred):
    logits, labels = eval_pred
    preds = np.argmax(logits, axis=-1)
    return {"macro_f1": f1_score(labels, preds, average="macro"), "accuracy": accuracy_score(labels, preds)}


@torch.no_grad()
def run_classifier(classifier, tokenizer, texts, batch_size=32, embeddings=False):
    """Class probabilities (and optionally mean-pooled last hidden states) for a list of texts."""
    classifier.eval()
    order = np.argsort([len(t) for t in texts])           # similar lengths in one batch = less padding
    probs, vectors = [None] * len(texts), [None] * len(texts)
    for i in range(0, len(texts), batch_size):
        idx = order[i:i + batch_size]
        batch = tokenizer([texts[j] for j in idx], truncation=True, max_length=MAX_LENGTH,
                          padding=True, return_tensors="pt").to(classifier.device)
        with torch.autocast("cuda", dtype=COMPUTE_DTYPE):
            output = classifier(**batch, output_hidden_states=embeddings)
        batch_probs = output.logits.float().softmax(-1).cpu().numpy()
        if embeddings:
            mask = batch["attention_mask"].unsqueeze(-1).float()
            pooled = (output.hidden_states[-1].float() * mask).sum(1) / mask.sum(1)
            pooled = pooled.cpu().numpy()
        for k, j in enumerate(idx):
            probs[j] = batch_probs[k]
            if embeddings:
                vectors[j] = pooled[k]
    return (np.vstack(probs), np.vstack(vectors)) if embeddings else np.vstack(probs)


def finetune(short_name, hf_name):
    adapter_dir = os.path.join(FT_DRIVE_FOLDER, f"adapter_{short_name}")
    info_file = os.path.join(FT_DRIVE_FOLDER, f"train_info_{short_name}.json")
    tokenizer, classifier = load_classifier(hf_name)

    if USE_CACHE and os.path.exists(adapter_dir) and os.path.exists(info_file):
        classifier = PeftModel.from_pretrained(classifier, adapter_dir)
        with open(info_file) as f:
            info = json.load(f)
        print(f"Loaded fine-tuned adapter of {short_name} from Drive.")
    else:
        if USE_4BIT:
            classifier = prepare_model_for_kbit_training(
                classifier, use_gradient_checkpointing=True,
                gradient_checkpointing_kwargs={"use_reentrant": False})
        lora_config = LoraConfig(task_type=TaskType.SEQ_CLS, r=16, lora_alpha=32, lora_dropout=0.05,
                                 target_modules=LORA_TARGETS)
        classifier = get_peft_model(classifier, lora_config)
        trainable, total = classifier.get_nb_trainable_parameters()
        print(f"Trainable parameters: {trainable:,} of {total:,} ({trainable / total:.2%})")

        # 5% of the steps as warm-up (as a number of steps: "warmup_ratio" no longer exists in transformers 5)
        steps_per_epoch = int(np.ceil(len(train_l) / (BATCH_SIZE * GRAD_ACCUM)))
        args = TrainingArguments(
            output_dir=f"checkpoints_{short_name}",
            num_train_epochs=EPOCHS,
            learning_rate=LEARNING_RATE,
            per_device_train_batch_size=BATCH_SIZE,
            per_device_eval_batch_size=BATCH_SIZE * 2,
            gradient_accumulation_steps=GRAD_ACCUM,
            lr_scheduler_type="cosine",
            warmup_steps=max(1, int(0.05 * steps_per_epoch * EPOCHS)),
            weight_decay=0.01,
            eval_strategy="epoch",
            save_strategy="epoch",
            save_total_limit=1,
            load_best_model_at_end=True,
            metric_for_best_model="macro_f1",
            gradient_checkpointing=True,
            gradient_checkpointing_kwargs={"use_reentrant": False},
            bf16=COMPUTE_DTYPE == torch.bfloat16,
            fp16=COMPUTE_DTYPE == torch.float16,
            logging_steps=20,
            report_to="none",
            seed=RANDOM_STATE,
        )
        trainer = WeightedTrainer(
            model=classifier, args=args,
            train_dataset=to_dataset(train_l, tokenizer), eval_dataset=to_dataset(val_l, tokenizer),
            data_collator=DataCollatorWithPadding(tokenizer), compute_metrics=compute_metrics,
        )
        torch.cuda.reset_peak_memory_stats()
        start = time.time()
        trainer.train()
        info = {
            "train_min": round((time.time() - start) / 60, 1),
            "peak_gpu_gb": round(torch.cuda.max_memory_allocated() / 1e9, 2),
            "params": f"{total / 1e9:.2f}B",
            "trainable_params": trainable,
            "log_history": trainer.state.log_history,
        }
        classifier = trainer.model
        classifier.save_pretrained(adapter_dir)
        with open(info_file, "w") as f:
            json.dump(info, f)
        shutil.rmtree(f"checkpoints_{short_name}", ignore_errors=True)
        print(f"Training: {info['train_min']} min, peak GPU memory {info['peak_gpu_gb']} GB")

    start = time.time()
    probs = run_classifier(classifier, tokenizer, test_l["input_text"].tolist())
    infer_time = time.time() - start
    report(f"{short_name} QLoRA", test_l["label_id"], probs.argmax(1),
           train_min=info["train_min"], infer_s_per_100=round(infer_time / len(test_l) * 100, 2),
           params=info["params"], trainable_params=info["trainable_params"], peak_gpu_gb=info["peak_gpu_gb"])
    return info


train_infos = {}
for short_name, hf_name in MODELS.items():
    train_infos[short_name] = finetune(short_name, hf_name)
    clear_gpu()

The training curves show whether the models still improve in the last epoch (then more epochs could help) or whether the validation score already decreases (overfitting).

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(13, 4))
for short_name, info in train_infos.items():
    history = pd.DataFrame(info["log_history"])
    train_loss = history.dropna(subset=["loss"]) if "loss" in history else pd.DataFrame()
    evals = history.dropna(subset=["eval_macro_f1"]) if "eval_macro_f1" in history else pd.DataFrame()
    if len(train_loss):
        axes[0].plot(train_loss["epoch"], train_loss["loss"], label=short_name)
    if len(evals):
        axes[1].plot(evals["epoch"], evals["eval_macro_f1"], marker="o", label=short_name)
axes[0].set(title="Training loss", xlabel="Epoch", ylabel="Weighted cross-entropy")
axes[1].set(title="Validation macro-F1", xlabel="Epoch", ylabel="Macro-F1")
for ax in axes:
    ax.legend()
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

### 9.8 Comparison of all models

The table compares quality and costs of all models on the same 300 test articles:

- `macro_f1`, `accuracy`: quality on the test set
- `params`, `trainable_params`: size of the model and number of trained parameters
- `train_min`: training time in minutes (0 for zero-shot)
- `infer_s_per_100`: time to classify 100 articles in seconds
- `peak_gpu_gb`: maximum GPU memory during training (fine-tuning) or inference (zero-shot)

The times depend on the GPU (L4 or A100) and should be compared within one run.

In [ ]:
comparison = pd.DataFrame(results).set_index("model")
display(comparison)

fig, ax = plt.subplots(figsize=(9, 4))
comparison["macro_f1"].sort_values().plot.barh(ax=ax, color="#3D7C83")
ax.set(title="Test macro-F1 of all models", xlabel="Macro-F1")
ax.grid(axis="x", alpha=0.3)
plt.tight_layout()
plt.show()

best_name = comparison["macro_f1"].idxmax()
print(f"Best model: {best_name}")
print(classification_report(test_l["label_id"], predictions[best_name], labels=range(len(LABELS)),
                            target_names=LABELS, zero_division=0))
fig, ax = plt.subplots(figsize=(8, 7))
ConfusionMatrixDisplay.from_predictions(test_l["label_id"], predictions[best_name], labels=range(len(LABELS)),
                                        display_labels=LABELS, xticks_rotation=45, ax=ax, colorbar=False)
ax.set_title(f"Confusion matrix: {best_name}")
plt.tight_layout()
plt.show()

### 9.9 Embeddings of the fine-tuned model

Besides the extrinsic task, we compare the representations intrinsically, as in section 8: for the test articles we compute a document vector with Word2Vec (averaged word vectors) and with the fine-tuned model (average of the last hidden states over all tokens), and measure with the governance labels

- the silhouette score of the five classes (centered vectors, cosine distance),
- the macro-F1 of a logistic regression with 5-fold cross-validation on the vectors.

The fine-tuned model was trained on exactly these classes, so a better separation is expected; the comparison shows how strongly the fine-tuning reorganises the representation space compared with the unsupervised Word2Vec vectors. The t-SNE plots show the same comparison visually.

In [ ]:
FT_MODEL = "Qwen3-1.7B"     # fine-tuned model for the embedding analysis and the trend analysis

tokenizer, classifier = load_classifier(MODELS[FT_MODEL])
classifier = PeftModel.from_pretrained(classifier, os.path.join(FT_DRIVE_FOLDER, f"adapter_{FT_MODEL}"))
_, llm_vectors = run_classifier(classifier, tokenizer, test_l["input_text"].tolist(), embeddings=True)
w2v_vectors = w2v_features(test_l)

rows, coords = [], {}
y_test = test_l["gold_label"].to_numpy()
for name, vectors in [("Word2Vec", w2v_vectors), (f"{FT_MODEL} fine-tuned", llm_vectors)]:
    centered = vectors - vectors.mean(axis=0)
    rows.append({
        "representation": name,
        "dimensions": vectors.shape[1],
        "silhouette": round(silhouette_score(centered, y_test, metric="cosine"), 3),
        "logreg_macro_f1_cv": round(cross_val_score(LogisticRegression(max_iter=3000, class_weight="balanced"),
                                                    centered, y_test, cv=5, scoring="f1_macro").mean(), 3),
    })
    coords[name] = TSNE(n_components=2, perplexity=30, metric="cosine", init="pca",
                        random_state=RANDOM_STATE).fit_transform(centered)
display(pd.DataFrame(rows))

fig, axes = plt.subplots(1, 2, figsize=(15, 6))
for ax, (name, xy) in zip(axes, coords.items()):
    for label in LABELS:
        mask = y_test == label
        ax.scatter(xy[mask, 0], xy[mask, 1], s=10, alpha=0.7, label=label)
    ax.set_title(f"{name} (t-SNE, test articles)")
    ax.set_xticks([])
    ax.set_yticks([])
axes[1].legend(bbox_to_anchor=(1.02, 1), loc="upper left", fontsize=8)
plt.tight_layout()
plt.show()

### 9.10 Trend analysis of the governance subtopics

Finally we use the fine-tuned model to classify **all articles of the corpus** and analyse how the governance subtopics develop between September 2024 and August 2026. In contrast to the keyword filter of Stage 1, the model distinguishes the subtopics and excludes articles that only mention governance terms.

We use the smaller model (`FT_MODEL`), because it has to classify about 47,000 articles (about 30-45 minutes on an L4 GPU). If the larger model is clearly better in section 9.8, `FT_MODEL` can be changed. The predictions are stored in Drive in parts of 2,000 articles, so an interrupted run continues where it stopped. With `CORPUS_SAMPLE` a random sample can be classified first to test the cell.

In [ ]:
CORPUS_SAMPLE = None      # e.g. 2000 for a quick test, None = all articles
CHUNK_SIZE = 2000
PRED_FILE = os.path.join(FT_DRIVE_FOLDER, f"corpus_predictions_{FT_MODEL}.csv")

corpus = df if CORPUS_SAMPLE is None else df.sample(n=CORPUS_SAMPLE, random_state=RANDOM_STATE)
corpus = corpus.assign(input_text=corpus.apply(model_input, axis=1))

done_urls = set(pd.read_csv(PRED_FILE)["url"]) if os.path.exists(PRED_FILE) else set()
todo = corpus[~corpus["url"].isin(done_urls)]
print(f"Already classified: {len(done_urls)}, to classify now: {len(todo)}")

for start in tqdm(range(0, len(todo), CHUNK_SIZE), desc="Corpus chunks"):
    chunk = todo.iloc[start:start + CHUNK_SIZE]
    probs = run_classifier(classifier, tokenizer, chunk["input_text"].tolist())
    out = pd.DataFrame({"url": chunk["url"].to_numpy(),
                        "pred_label": [ID2LABEL[i] for i in probs.argmax(1)],
                        "pred_prob": probs.max(1).round(3)})
    out.to_csv(PRED_FILE, mode="a", header=not os.path.exists(PRED_FILE), index=False)

corpus_pred = pd.read_csv(PRED_FILE).drop_duplicates("url").set_index("url")
corpus = corpus.join(corpus_pred, on="url")
del classifier
clear_gpu()

print(corpus["pred_label"].value_counts(normalize=True).round(3))
print(f"\nGovernance articles according to the model: {(corpus['pred_label'] != 'NOT_GOVERNANCE').mean():.1%}, "
      f"according to the Stage 1 keyword filter: {corpus['stage1_governance'].mean():.1%}")

In [ ]:
GOV_LABELS = [label for label in LABELS if label != "NOT_GOVERNANCE"]

monthly_share = (
    pd.crosstab(corpus["month"], corpus["pred_label"], normalize="index")
    .reindex(columns=GOV_LABELS, fill_value=0) * 100
)
monthly_share.index = pd.to_datetime(monthly_share.index)

fig, axes = plt.subplots(2, 1, figsize=(13, 9), sharex=True)
monthly_share.plot(kind="area", stacked=True, alpha=0.85, ax=axes[0])
axes[0].set(title=f"Share of AI Governance articles per month ({FT_MODEL} fine-tuned)",
            ylabel="% of all articles of the month")
axes[0].legend(title="Subtopic", bbox_to_anchor=(1.02, 1), loc="upper left")

monthly_share.plot(ax=axes[1], marker="o", linewidth=1.5)
axes[1].set(title="Subtopics separately", xlabel="Month", ylabel="% of all articles of the month")
axes[1].legend().remove()
for ax in axes:
    ax.grid(alpha=0.3)
plt.tight_layout()
plt.show()

trend = pd.DataFrame({
    "overall_pct": monthly_share.mean(),
    "first_6_months_pct": monthly_share.iloc[:6].mean(),
    "last_6_months_pct": monthly_share.iloc[-6:].mean(),
})
trend["change_pp"] = trend["last_6_months_pct"] - trend["first_6_months_pct"]
trend["peak_month"] = monthly_share.idxmax().dt.strftime("%Y-%m")
display(trend.round(2))